# Task 3 — Step 1: Standard Group Relative Policy Optimization (GRPO)

### Objective & Formulation
GRPO removes the learned value critic $V_\phi(s)$ entirely, replacing it with group-relative advantage normalization across $K$ sampled completions per prompt:
$$A_k = \frac{r_k - \mu_r}{\sigma_r + \varepsilon}, \quad \mu_r = \frac{1}{K}\sum_{j=1}^K r_j, \quad \sigma_r = \sqrt{\frac{1}{K}\sum_{j=1}^K (r_j - \mu_r)^2}$$

The policy is optimized using the clipped surrogate loss with sequence normalization:
$$\mathcal{L}_{\text{GRPO}}(\theta) = -\hat{\mathbb{E}}_{x,\{y_k\}} \left[ \frac{1}{K}\sum_{k=1}^K \frac{1}{T_k}\sum_{t=1}^{T_k} \min\left(\rho_{k,t} A_k, \text{clip}(\rho_{k,t}, 1-\epsilon, 1+\epsilon)A_k\right) \right] + \beta D_{\text{KL}}(\pi_\theta \parallel \pi_{\text{ref}})$$

### Required Evidence:
1. **Continuation Trajectories**: Reward, KL drift, within-group reward standard deviation, uninformative-group fraction ($\sigma_r = 0$), entropy, and length.
2. **Compute Comparison**: Peak VRAM and wall-clock time compared directly to Task 2 PPO.


In [ ]:
%matplotlib inline
import os, sys, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Configure matplotlib style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.labelsize'] = 11

# Ensure project root is in sys.path
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
try:
    os.chdir(ROOT)
except Exception:
    pass

print("Project root:", ROOT)


## 1. Run GRPO Continuation or Load Records


In [ ]:
FORCE_RUN = False
if FORCE_RUN:
    !python -m task3_grpo.continue_train --config configs/grpo.yaml --run-name standard
    !python -m task3_grpo.evaluate --config configs/grpo.yaml --adapter outputs/task3_grpo/standard --name standard
else:
    print("[OK] Loading existing GRPO continuation records...")


## 2. Summary Table of GRPO Continuation Run


In [ ]:
summary_file = "results/task3_grpo/grpo_summary_standard.json"
if os.path.exists(summary_file):
    with open(summary_file, encoding='utf-8') as f:
        summary_data = json.load(f)
    display(pd.DataFrame([summary_data]))


## 3. Visualizing GRPO Trajectories & Group Informativeness


In [ ]:
log_file = "results/task3_grpo/grpo_train_standard.jsonl"
if os.path.exists(log_file):
    records = [json.loads(line) for line in open(log_file, encoding='utf-8')]
    df = pd.DataFrame(records)
    
    fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
    
    # 1. Reward
    axes[0].plot(df['update'], df['reward_mean'], color='#10B981', lw=2)
    axes[0].set_title('Mean Reward Trajectory')
    axes[0].set_xlabel('Update')
    axes[0].grid(True, alpha=0.3)
    
    # 2. Within-Group Std Dev
    axes[1].plot(df['update'], df['group_std_mean'], color='#3B82F6', lw=2)
    axes[1].set_title('Within-Group Reward Std Dev')
    axes[1].set_xlabel('Update')
    axes[1].grid(True, alpha=0.3)
    
    # 3. Uninformative Groups %
    axes[2].plot(df['update'], df['uninformative_group_fraction']*100, color='#EF4444', lw=2)
    axes[2].set_title('Zero-Variance (Uninformative) Groups (%)')
    axes[2].set_xlabel('Update')
    axes[2].grid(True, alpha=0.3)
    
    # 4. KL Drift
    axes[3].plot(df['update'], df['kl_mean'], color='#F59E0B', lw=2)
    axes[3].set_title('Reference Policy Drift (KL)')
    axes[3].set_xlabel('Update')
    axes[3].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


## 4. Compute Comparison: PPO vs. GRPO
- **Memory Footprint**: How much VRAM was saved by discarding the value model critic ($V_\phi$)?
- **Wall-Clock Time**: Did sampling $K=4$ completions per prompt balance out the removal of the critic forward/backward pass?
